# CPSTN Enhanced — PEMS-BAY Testing / Evaluation (FIXED)

Use this notebook **after training**.

It loads the exact checkpoint, uses the checkpoint's saved scaler and graph, reconstructs the exact architecture, and evaluates only the chronological test segment.

Outputs:
- overall MAE / RMSE / MAPE
- 5/10/15/20/25/30-minute horizon metrics
- persistence baseline
- model-vs-baseline comparison
- forecast plot
- CSV reports

The test set is never used to train or select the checkpoint.

In [ ]:
# 1. Environment
!nvidia-smi
!pip -q install pandas numpy matplotlib tqdm

import os, pickle, urllib.request, hashlib, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab: Runtime -> Change runtime type -> GPU.")
device = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Download + verify the same PEMS-BAY distribution
DATA_DIR = Path("/content/pems_bay")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://zenodo.org/records/5146275/files"
DATA_URL = f"{BASE}/PEMS-BAY.csv?download=1"
ADJ_URL = f"{BASE}/adj_mx_bay.pkl?download=1"

CSV_PATH = DATA_DIR / "PEMS-BAY.csv"
ADJ_PATH = DATA_DIR / "adj_mx_bay.pkl"

EXPECTED_MD5 = {
    "PEMS-BAY.csv": "c8dea58987a5882e946217c22fdb8256",
    "adj_mx_bay.pkl": "55d25daceac847312b7748c59ded6f77",
}

def md5(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def download_verified(url, path):
    expected = EXPECTED_MD5[path.name]
    if path.exists() and path.stat().st_size > 0 and md5(path) == expected:
        print("Verified existing:", path.name)
        return
    if path.exists(): path.unlink()
    print("Downloading:", path.name)
    urllib.request.urlretrieve(url, path)
    actual = md5(path)
    if actual != expected:
        raise RuntimeError(f"MD5 mismatch for {path.name}: {actual} != {expected}")

download_verified(DATA_URL, CSV_PATH)
download_verified(ADJ_URL, ADJ_PATH)

In [ ]:
# 3. Upload checkpoint
MODEL_PATH = Path("/content/CPSTN_Enhanced_PEMS_BAY_best.pt")

if not MODEL_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    pt_files = [name for name in uploaded if name.endswith((".pt", ".pth"))]
    if not pt_files:
        raise RuntimeError("Upload CPSTN_Enhanced_PEMS_BAY_best.pt")
    shutil.move(pt_files[0], MODEL_PATH)

print("Checkpoint:", MODEL_PATH)

In [ ]:
# 4. Load checkpoint FIRST and use its saved preprocessing configuration
try:
    ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(MODEL_PATH, map_location=device)

required = ["model_state_dict","mean","std","adjacency","input_steps","horizon","num_nodes","input_channels","hidden"]
missing = [k for k in required if k not in ckpt]
if missing:
    raise RuntimeError("Checkpoint missing fields: " + str(missing))

INPUT_STEPS = int(ckpt["input_steps"])
HORIZON = int(ckpt["horizon"])
NUM_NODES = int(ckpt["num_nodes"])
INPUT_CHANNELS = int(ckpt["input_channels"])
HIDDEN = int(ckpt["hidden"])

if NUM_NODES != 325 or INPUT_CHANNELS != 3:
    raise RuntimeError(f"Unexpected checkpoint config: nodes={NUM_NODES}, channels={INPUT_CHANNELS}")

mean = np.asarray(ckpt["mean"], dtype=np.float32)
std = np.asarray(ckpt["std"], dtype=np.float32)
A_norm = np.asarray(ckpt["adjacency"], dtype=np.float32)

if mean.shape != (325,) or std.shape != (325,) or A_norm.shape != (325,325):
    raise RuntimeError("Checkpoint scaler/adjacency shapes are invalid.")

A_tensor = torch.from_numpy(A_norm).to(device)
print("Checkpoint epoch:", ckpt.get("epoch"))
print("Best validation loss:", ckpt.get("best_val_loss"))
print("Input/Horizon:", INPUT_STEPS, "->", HORIZON)

In [ ]:
# 5. Load data and construct the SAME chronological split
df = pd.read_csv(CSV_PATH, index_col=0)
df.index = pd.to_datetime(df.index)

if df.shape[1] != 325:
    raise ValueError(f"Expected 325 sensors, got {df.shape[1]}")
if not df.index.is_monotonic_increasing:
    raise ValueError("Timestamps are not sorted.")

df = df.apply(pd.to_numeric, errors="coerce")
df = df.interpolate(method="linear", limit_direction="both").ffill().bfill()

if df.isna().any().any():
    raise RuntimeError("NaNs remain after preprocessing.")

speed = df.to_numpy(dtype=np.float32)
timestamps = df.index
T = len(speed)

train_end = int(T * 0.70)
val_end = int(T * 0.80)

# IMPORTANT: use the TRAIN scaler stored in the checkpoint.
speed_z = ((speed - mean) / np.maximum(std, 1e-3)).astype(np.float32)

minutes = timestamps.hour.to_numpy() * 60 + timestamps.minute.to_numpy()
tod = (minutes / (24 * 60)).astype(np.float32)
time_features = np.stack([
    np.sin(2*np.pi*tod),
    np.cos(2*np.pi*tod)
], axis=1).astype(np.float32)

print("Test interval:", timestamps[val_end], "->", timestamps[-1])

In [ ]:
# 6. Test windows
class TrafficWindowDataset(Dataset):
    def __init__(self, speed_z, time_features, start, end, input_steps, horizon):
        self.speed = speed_z
        self.time = time_features
        self.start = int(start)
        self.end = int(end)
        self.input_steps = int(input_steps)
        self.horizon = int(horizon)
        self.first_t = max(self.start, self.input_steps)
        self.last_t = self.end - self.horizon

    def __len__(self):
        return self.last_t - self.first_t + 1

    def __getitem__(self, i):
        t = self.first_t + i
        s = self.speed[t-self.input_steps:t]
        tf = self.time[t-self.input_steps:t]
        tf = np.broadcast_to(tf[:, None, :], (self.input_steps, s.shape[1], 2))
        x = np.concatenate([s[..., None], tf], axis=-1).astype(np.float32)
        y = self.speed[t:t+self.horizon].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)

test_ds = TrafficWindowDataset(speed_z, time_features, val_end, T, INPUT_STEPS, HORIZON)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print("Test windows:", len(test_ds))

In [ ]:
# 7. Exact model architecture
class GatedTemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dropout=0.10):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels*2, (kernel_size,1),
                              padding=(kernel_size//2,0))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h, g = self.conv(x).chunk(2, dim=1)
        return self.dropout(h * torch.sigmoid(g))

class GraphConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, 1)
    def forward(self, x, A):
        return self.proj(torch.einsum("ij,bctj->bcti", A, x))

class EnhancedSTGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.10):
        super().__init__()
        self.temporal1 = GatedTemporalConv(in_channels, out_channels, 3, dropout)
        self.graph = GraphConv(out_channels, out_channels)
        self.temporal2 = GatedTemporalConv(out_channels, out_channels, 3, dropout)
        self.residual = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.norm = nn.GroupNorm(1, out_channels)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, A):
        r = self.residual(x)
        x = F.gelu(self.temporal1(x))
        x = F.gelu(self.graph(x, A))
        x = self.temporal2(x)
        x = self.dropout(self.norm(x))
        return F.gelu(x + r)

class EnhancedSTGCN(nn.Module):
    def __init__(self, num_nodes=325, input_channels=3, hidden=64, horizon=6, dropout=0.10):
        super().__init__()
        self.input_proj = nn.Conv2d(input_channels, hidden, 1)
        self.block1 = EnhancedSTGCNBlock(hidden, hidden, dropout)
        self.block2 = EnhancedSTGCNBlock(hidden, 96, dropout)
        self.block3 = EnhancedSTGCNBlock(96, 96, dropout)
        self.block4 = EnhancedSTGCNBlock(96, hidden, dropout)
        self.head = nn.Sequential(
            nn.Conv2d(hidden, hidden, 1), nn.GELU(), nn.Dropout(dropout),
            nn.Conv2d(hidden, horizon, 1)
        )
    def forward(self, x, A):
        x = x.permute(0,3,1,2).contiguous()
        x = self.input_proj(x)
        x = self.block1(x,A)
        x = self.block2(x,A)
        x = self.block3(x,A)
        x = self.block4(x,A)
        return self.head(x[:,:,-1:,:]).squeeze(2)

model = EnhancedSTGCN(NUM_NODES, INPUT_CHANNELS, HIDDEN, HORIZON).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Model loaded successfully.")

In [ ]:
# 8. Full test inference
pred_z, true_z = [], []

with torch.no_grad():
    for X, Y in tqdm(test_loader, desc="Testing"):
        P = model(X.to(device, non_blocking=True), A_tensor)
        pred_z.append(P.cpu().numpy())
        true_z.append(Y.numpy())

pred_z = np.concatenate(pred_z)
true_z = np.concatenate(true_z)

pred = pred_z * std[None,None,:] + mean[None,None,:]
true = true_z * std[None,None,:] + mean[None,None,:]

assert pred.shape == true.shape
print("Prediction/target shape:", pred.shape)

In [ ]:
# 9. Overall metrics
def metrics(p, y):
    err = p-y
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    mape = np.mean(np.abs(err) / np.maximum(np.abs(y), 1.0)) * 100
    return float(mae), float(rmse), float(mape)

mae, rmse, mape = metrics(pred, true)

print("="*60)
print(f"Enhanced STGCN | MAE : {mae:.4f} mph")
print(f"Enhanced STGCN | RMSE: {rmse:.4f} mph")
print(f"Enhanced STGCN | MAPE: {mape:.2f}%")
print("="*60)

In [ ]:
# 10. Horizon metrics
rows = []
for h in range(HORIZON):
    h_mae, h_rmse, h_mape = metrics(pred[:,h,:], true[:,h,:])
    rows.append({
        "horizon_min": (h+1)*5,
        "MAE_mph": h_mae,
        "RMSE_mph": h_rmse,
        "MAPE_percent": h_mape
    })
horizon_df = pd.DataFrame(rows)
display(horizon_df)

In [ ]:
# 11. Persistence baseline
# Last observed speed -> repeated across all future steps.
# Build it directly from the input windows to avoid any indexing ambiguity.
baseline_parts = []
target_parts = []
for X, Y in test_loader:
    baseline_parts.append(np.repeat(X[:, -1, :, 0].numpy()[:,None,:], HORIZON, axis=1))
    target_parts.append(Y.numpy())

baseline_z = np.concatenate(baseline_parts)
baseline = baseline_z * std[None,None,:] + mean[None,None,:]
base_true_z = np.concatenate(target_parts)
base_true = base_true_z * std[None,None,:] + mean[None,None,:]

b_mae, b_rmse, b_mape = metrics(baseline, base_true)

comparison = pd.DataFrame({
    "model": ["Persistence", "Enhanced STGCN"],
    "MAE_mph": [b_mae, mae],
    "RMSE_mph": [b_rmse, rmse],
    "MAPE_percent": [b_mape, mape]
})
display(comparison)
print("MAE improvement vs persistence:",
      round((b_mae-mae)/b_mae*100, 2), "%")

In [ ]:
# 12. Forecast visualization
sample_idx = min(100, len(pred)-1)
sensor_idx = 0

t0 = val_end + sample_idx
hist = speed[t0-INPUT_STEPS:t0, sensor_idx]
actual = true[sample_idx,:,sensor_idx]
forecast = pred[sample_idx,:,sensor_idx]

x_hist = np.arange(-INPUT_STEPS,0)
x_future = np.arange(0,HORIZON)

plt.figure(figsize=(11,5))
plt.plot(x_hist, hist, label="Observed history")
plt.plot(x_future, actual, marker="o", label="Actual")
plt.plot(x_future, forecast, marker="x", label="Enhanced STGCN")
plt.axvline(0, linestyle="--", alpha=0.5)
plt.xlabel("5-minute steps")
plt.ylabel("Speed (mph)")
plt.title("PEMS-BAY Forecast — Sensor 0")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# 13. Save reports
report = pd.DataFrame({
    "metric": ["MAE_mph","RMSE_mph","MAPE_percent",
               "Persistence_MAE_mph","Persistence_RMSE_mph","Persistence_MAPE_percent"],
    "value": [mae,rmse,mape,b_mae,b_rmse,b_mape]
})
report_path = "/content/CPSTN_Enhanced_PEMS_BAY_test_report.csv"
horizon_path = "/content/CPSTN_Enhanced_PEMS_BAY_horizon_metrics.csv"
report.to_csv(report_path,index=False)
horizon_df.to_csv(horizon_path,index=False)
print(report_path)
print(horizon_path)

In [ ]:
# 14. Download reports
from google.colab import files
files.download("/content/CPSTN_Enhanced_PEMS_BAY_test_report.csv")
files.download("/content/CPSTN_Enhanced_PEMS_BAY_horizon_metrics.csv")